# Tusker — Qwen3-Embedding-0.6B Embeddings

Generates dense embeddings for `gl_news_data_canonical.jsonl` using
[Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).

**Pipeline**
1. Mount G1oogle Drive
2. Load the JSONL dataset
3. Process in configurable batches — each batch checkpoint is written to Drive before moving on
4. Merge all checkpoints into a single Parquet file

**Resume support** — re-running the notebook skips batches whose checkpoint already exists on Drive.

**Runtime recommendation** — T4 GPU (free tier is fine; A100 is faster).

## 1 · Install dependencies

In [ ]:
%%capture
!pip install -q transformers==4.51.3 accelerate einops

## 2 · Configuration

In [ ]:
import os

# ── Google Drive paths ──────────────────────────────────────────────────────
DRIVE_ROOT        = "/content/drive/MyDrive"
DRIVE_PROJECT_DIR = os.path.join(DRIVE_ROOT, "news-hunter", "embeddings")
DRIVE_JSONL_PATH  = os.path.join(DRIVE_ROOT, "news-hunter", "dataset", "gl_news_data_canonical.jsonl")
CHECKPOINT_DIR    = os.path.join(DRIVE_PROJECT_DIR, "checkpoints")
OUTPUT_PATH       = os.path.join(DRIVE_PROJECT_DIR, "gl_news_embeddings.parquet")

# ── Model ───────────────────────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen3-Embedding-0.6B"   # HuggingFace weights used for generation
MODEL_NAME = "qwen3-embedding:0.6b"        # canonical name stored in DB / used at query time
MAX_LENGTH = 512   # tokens per document; Qwen3-Emb supports up to 32 768

# ── Text fields to embed (concatenated in order) ────────────────────────────
# Set EMBED_FULL_CONTENT=True to include the full article body.
# Full content gives richer vectors but uses more VRAM and is slower.
EMBED_FULL_CONTENT = False

# ── Batching ────────────────────────────────────────────────────────────────
BATCH_SIZE = 64    # documents per forward pass (lower if OOM)

print("Config OK")
print(f"  dataset : {DRIVE_JSONL_PATH}")
print(f"  model   : {MODEL_ID}")
print(f"  output  : {OUTPUT_PATH}")
print(f"  ckpt dir: {CHECKPOINT_DIR}")

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Drive mounted, checkpoint dir ready:", CHECKPOINT_DIR)

## 4 · Load dataset

The notebook expects the JSONL file already uploaded to Drive at `DRIVE_JSONL_PATH`.
If it is not there yet, run the cell below to upload it from your local machine.

In [ ]:
# ── Optional: upload from local machine ─────────────────────────────────────
# Skip this cell if the file is already on Drive.

if not os.path.exists(DRIVE_JSONL_PATH):
    from google.colab import files
    import shutil

    print("File not found on Drive — upload it now.")
    uploaded = files.upload()                          # opens a file picker
    fname    = next(iter(uploaded))
    dest_dir = os.path.dirname(DRIVE_JSONL_PATH)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.move(fname, DRIVE_JSONL_PATH)
    print(f"Saved to {DRIVE_JSONL_PATH}")
else:
    print(f"Dataset already on Drive: {DRIVE_JSONL_PATH}")

In [ ]:
import json

articles = []
with open(DRIVE_JSONL_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if line:
            articles.append(json.loads(line))

print(f"Loaded {len(articles):,} articles")
print("Sample keys:", list(articles[0].keys()))

## 5 · Build text inputs

In [ ]:
def build_text(article: dict) -> str:
    """Concatenate the fields we want to embed into a single string.

    Field priority
    --------------
    title       — always included
    description — always included
    full_content / content — only when EMBED_FULL_CONTENT=True
    """
    parts = [
        article.get("title", "").strip(),
        article.get("description", "").strip(),
    ]
    if EMBED_FULL_CONTENT:
        body = article.get("full_content") or article.get("content", "")
        parts.append(body.strip())

    return " ".join(p for p in parts if p)


doc_ids = [a["id"] for a in articles]
texts   = [build_text(a) for a in articles]

# Sanity-check
empty = sum(1 for t in texts if not t)
print(f"Texts built: {len(texts):,}  (empty: {empty})")
print("Example:", texts[0][:200])

## 6 · Load model

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model     = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()

EMBED_DIM = model.config.hidden_size
print(f"Model loaded — embedding dim: {EMBED_DIM}")

## 7 · Embedding helpers

In [ ]:
import numpy as np


def last_token_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Qwen3-Embedding uses last-token pooling (decoder architecture)."""
    # Find the position of the last real (non-padding) token per sample.
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return hidden_states[:, -1]
    seq_len = attention_mask.sum(dim=1) - 1          # 0-indexed last position
    batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
    return hidden_states[batch_idx, seq_len]


@torch.inference_mode()
def embed_batch(batch_texts: list[str]) -> np.ndarray:
    """Tokenize, forward-pass, pool, L2-normalise → (N, D) float32 numpy array."""
    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    outputs    = model(**encoded)
    embeddings = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().float().numpy()


def checkpoint_path(batch_idx: int) -> str:
    return os.path.join(CHECKPOINT_DIR, f"batch_{batch_idx:06d}.npz")


def save_checkpoint(batch_idx: int, ids: list[str], vectors: np.ndarray) -> None:
    # Ensure the checkpoint dir exists — resilient to running this cell after a
    # runtime restart without re-running the mount cell (Drive FileNotFoundError).
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    np.savez_compressed(
        checkpoint_path(batch_idx),
        ids=np.array(ids),
        embeddings=vectors,
    )


def checkpoint_exists(batch_idx: int) -> bool:
    return os.path.exists(checkpoint_path(batch_idx))


print("Helpers defined")

## 8 · Embed — batch loop with Drive checkpointing

In [ ]:
import math
import time

n_docs    = len(texts)
n_batches = math.ceil(n_docs / BATCH_SIZE)

# Count already-done batches (for resume)
done_count = sum(1 for i in range(n_batches) if checkpoint_exists(i))
print(f"Total batches : {n_batches}")
print(f"Already done  : {done_count}  (will skip)")
print(f"Remaining     : {n_batches - done_count}")
print()

t0 = time.time()

for batch_idx in range(n_batches):
    if checkpoint_exists(batch_idx):
        continue

    lo = batch_idx * BATCH_SIZE
    hi = min(lo + BATCH_SIZE, n_docs)

    batch_texts = texts[lo:hi]
    batch_ids   = doc_ids[lo:hi]

    vectors = embed_batch(batch_texts)
    save_checkpoint(batch_idx, batch_ids, vectors)

    elapsed  = time.time() - t0
    done_now = batch_idx + 1
    rate     = (done_now - done_count) / elapsed if elapsed > 0 else 0
    eta_s    = (n_batches - done_now) / rate if rate > 0 else float("inf")

    print(
        f"  batch {batch_idx + 1:>6}/{n_batches}  "
        f"docs [{lo:>7}–{hi:>7}]  "
        f"elapsed {elapsed:>6.0f}s  "
        f"ETA {eta_s / 60:>5.1f} min",
        end="\r",
        flush=True,
    )

print(f"\nAll {n_batches} batches done in {(time.time() - t0) / 60:.1f} min")

## 9 · Merge checkpoints → single Parquet

In [ ]:
import pyarrow as pa                # preinstalled on Colab
import pyarrow.parquet as pq
from datetime import datetime, timezone

all_ids        = []
all_embeddings = []

missing = []
for batch_idx in range(n_batches):
    path = checkpoint_path(batch_idx)
    if not os.path.exists(path):
        missing.append(batch_idx)
        continue
    ck = np.load(path, allow_pickle=False)
    all_ids.append(ck["ids"])
    all_embeddings.append(ck["embeddings"])

if missing:
    raise RuntimeError(f"Missing checkpoints for batches: {missing}. Re-run cell 8.")

ids_arr = np.concatenate(all_ids,        axis=0).astype(str)          # (N,)    str (uuid)
emb_arr = np.concatenate(all_embeddings, axis=0).astype(np.float32)   # (N, D)  float32
N, D = emb_arr.shape

print(f"Merged  — ids: {ids_arr.shape}  embeddings: {emb_arr.shape}")

# id: string, embedding: list<float32>  → read directly by parquet-go on ingest
table = pa.table({
    "id":        pa.array(ids_arr, type=pa.string()),
    "embedding": pa.array(list(emb_arr), type=pa.list_(pa.float32())),
})

# File-level metadata travels with the data (model_name must match query-time).
table = table.replace_schema_metadata({
    "model":              MODEL_NAME,
    "hf_model_id":        MODEL_ID,
    "dim":                str(D),
    "pooling":            "last_token",
    "normalized":         "l2",
    "max_length":         str(MAX_LENGTH),
    "embed_full_content": str(EMBED_FULL_CONTENT),
    "row_count":          str(N),
    "created_at":         datetime.now(timezone.utc).isoformat(),
})

print(f"Saving to {OUTPUT_PATH} …")
pq.write_table(table, OUTPUT_PATH, compression="zstd")

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved — {size_mb:.1f} MB  ({N:,} rows x {D} dims)")

## 10 · Quick verification

In [ ]:
import pyarrow.parquet as pq

t          = pq.read_table(OUTPUT_PATH)
ids        = t.column("id").to_pylist()
embeddings = np.array(t.column("embedding").to_pylist(), dtype=np.float32)

print("=== Verification ===")
print(f"Docs      : {len(ids):,}")
print(f"Emb shape : {embeddings.shape}")
print(f"Emb dtype : {embeddings.dtype}")
print(f"Metadata  : {t.schema.metadata}")
print()

# Norms should all be ~1.0 (L2-normalised)
norms = np.linalg.norm(embeddings, axis=1)
print(f"L2 norms  : mean={norms.mean():.4f}  std={norms.std():.6f}  min={norms.min():.4f}  max={norms.max():.4f}")
print()

# Cosine similarity between first two documents
cos_sim = float(np.dot(embeddings[0], embeddings[1]))
print(f"cos_sim(doc[0], doc[1]) = {cos_sim:.4f}")
print(f"doc[0] id : {ids[0]}")
print(f"doc[1] id : {ids[1]}")
print()
print("All checks passed ✓")

## Notes

### Output format

The final `gl_news_embeddings.parquet` has two columns plus file-level metadata.

| Column | Arrow type | Description |
|--------|-----------|-------------|
| `id` | `string` | UUID of each article (`id` field from JSONL) |
| `embedding` | `list<float32>` | 1024-dim L2-normalised dense vector |

**File metadata** (`table.schema.metadata`) — carried with the data and read by the
Go ingester so `model_name` always matches what was generated:

| Key | Example |
|-----|---------|
| `model` | `qwen3-embedding:0.6b` (canonical name stored in DB) |
| `hf_model_id` | `Qwen/Qwen3-Embedding-0.6B` |
| `dim` | `1024` |
| `pooling` | `last_token` |
| `normalized` | `l2` |
| `row_count` / `created_at` | provenance |

Load with:
```python
import pyarrow.parquet as pq
t = pq.read_table("gl_news_embeddings.parquet")
ids = t.column("id").to_pylist()
emb = t.column("embedding").to_numpy(zero_copy_only=False)
print(t.schema.metadata)
```

### Checkpoint layout (Drive)
```
MyDrive/news-hunter/embeddings/
  checkpoints/
    batch_000000.npz            ← intermediate, npz is fine here
    batch_000001.npz
    …
  gl_news_embeddings.parquet    ← final merged file (upload this to S3)
```

### Resume
If the Colab runtime disconnects, just **Run All** again. Cell 8 scans for existing checkpoint files and skips them.

### Embedding fields
Set `EMBED_FULL_CONTENT = True` in cell **2** to include full article body text (slower, richer vectors, ~2× VRAM usage).

### Model
`Qwen/Qwen3-Embedding-0.6B` — 596 M parameters, 1024-dim output, supports up to 32 768 tokens. Instruction-tuned for retrieval; no task prefix needed for document-side encoding.